In [26]:
import random
import os
import shutil
import albumentations as A
import numpy as np
import cv2 as cv
from PIL import Image
import matplotlib.pyplot as plt
import yaml

In [16]:
def change_extention(file_name,new_extention):
    file,_ = file_name.rsplit('.', 1)
    return file+f'.{new_extention}'


In [17]:
def get_annotations(label_path):
    with open(label_path, 'r') as f:
        lines = f.readlines()
    bboxes = []
    for line in lines:
        _, x_center, y_center, width, height = map(float, line.strip().split())
        bboxes.append([x_center, y_center, width, height])
    return bboxes
def save_annotations(bboxes,destination_path,class_index):
    annotations=""
    for bbox in bboxes:
       yolo_bbox=[class_index]+bbox
       annotations+=' '.join(str(num) for num in yolo_bbox)+'\n'
    with open(destination_path, "w") as file:
        file.write(annotations)


In [18]:
def apply_transform(source_paths,destination_paths,class_index,pipeline):
    image = cv.imread(source_paths[0])
    if not isinstance(image, np.ndarray):
        return False
    
    try:
        bboxes = get_annotations(source_paths[1])
    except FileNotFoundError:
        return False    
    transformed = pipeline(image=image, bboxes=bboxes, class_labels=[class_index] * len(bboxes))
    transformed_image = transformed["image"]
    transformed_bboxes = transformed["bboxes"]
    cv.imwrite(destination_paths[0],transformed_image)
    save_annotations(transformed_bboxes,destination_paths[1],class_index)
    return True


In [19]:
TARGET_HEIGHT=512
augmentation_pipeline = A.Compose([
    A.LongestMaxSize(max_size=TARGET_HEIGHT, interpolation=cv.INTER_LINEAR),
    A.PadIfNeeded(
      min_height=TARGET_HEIGHT,
      min_width=TARGET_HEIGHT,
      border_mode=cv.BORDER_CONSTANT,
      value=(255, 255, 255),),
    A.Affine(
        scale=(0.5, 1.5),
        translate_percent=(0.2, 0.2),
        rotate=0.2,
        shear=0.2,
        cval=(255, 255, 255),
        p=1.0,
    ),
    A.Blur(p=0.01),
    A.MedianBlur(p=0.01),
    A.ToGray(p=0.01),
    A.CLAHE(p=0.01),
    A.ColorJitter(
        contrast=0.1,
        saturation=0.6,
        hue=0.015,
        brightness=0.4,
    ),  
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),

  ],bbox_params = A.BboxParams(
        format="yolo",
        label_fields=["class_labels"]
    ))
resize_pipeline = A.Compose([
    A.LongestMaxSize(max_size=TARGET_HEIGHT, interpolation=cv.INTER_LINEAR),
    A.PadIfNeeded(
      min_height=TARGET_HEIGHT,
      min_width=TARGET_HEIGHT,
      border_mode=cv.BORDER_CONSTANT,
      value=(255, 255, 255),),
  ],bbox_params = A.BboxParams(
        format="yolo",
        label_fields=["class_labels"]
    ))

In [ ]:
def export_dataset(source_directory,destination_directory,rename_pattern,number_of_samples_needed,validation):
    os.makedirs(destination_directory+'\\images\\val')
    os.makedirs(destination_directory+'\\images\\train')
    os.makedirs(destination_directory+'\\labels\\val')
    os.makedirs(destination_directory+'\\labels\\train')

    counter = class_index= 0
    val_counter=train_counter=0
    classes=[]
    for images_root, _, image_files in os.walk(source_directory+"\\train"):
        labels_root=images_root.replace('train','labels')
        _,class_name=images_root.rsplit('\\',1)

        if(class_name=='train'):
            continue
        
        number_of_samples=len(os.listdir(images_root))
        

        random_numbers = random.sample(range(number_of_samples_needed*class_index, number_of_samples_needed*(class_index+1)+1), int(number_of_samples_needed*validation))
        random_numbers.sort()
        full_iterations=int(np.ceil(number_of_samples_needed/number_of_samples))
        classes.append(class_name)
         
        for iteration in range(full_iterations):
            for image_file in image_files:
                if counter>=number_of_samples_needed*(class_index+1):
                    break
                if image_file.lower().endswith(('.png', '.jpg', '.jpeg')):
                    img_src_path = os.path.join(images_root, image_file)
                    lbl_src_path = os.path.join(labels_root, change_extention(image_file,'txt'))
                    
                    if random_numbers!=[] and counter==random_numbers[0]:
                        new_name = rename_pattern.format(count=val_counter,extention='jpg', original_filename=image_file)
                        img_dst_path = os.path.join(destination_directory+'\\images\\val', new_name)
                        lbl_dst_path = os.path.join(destination_directory+'\\labels\\val', change_extention(new_name,'txt'))
                        saved=apply_transform([img_src_path,lbl_src_path],[img_dst_path,lbl_dst_path],class_index,resize_pipeline if iteration==0 else augmentation_pipeline)
                        if saved==True:
                            random_numbers.pop(0)
                            val_counter+=1
                            counter+=1
                    else:
                        new_name = rename_pattern.format(count=train_counter,extention='jpg', original_filename=image_file)
                        img_dst_path = os.path.join(destination_directory+'\\images\\train', new_name)
                        lbl_dst_path = os.path.join(destination_directory+'\\labels\\train', change_extention(new_name,'txt'))
                        saved=apply_transform([img_src_path,lbl_src_path],[img_dst_path,lbl_dst_path],class_index,resize_pipeline if iteration==0 else augmentation_pipeline)
                        if saved==True:
                            train_counter+=1
                            counter+=1       
        class_index+=1
        # if class_index==3:
        #     return
    data = {
        'path': destination_directory,
        'train': 'images/train',  
        'val': 'images/val',  
        'nc': class_index+1 ,
        'names': classes,

        # Model
        'model': 'yolov11n.yaml',

        # Hyperparameters
        'epochs': 100,
        'batch_size': 16,
        'lr0': 0.01, 
    }
    with open(destination_directory+'\\data.yaml', 'w') as file:
        yaml.dump(data, file)
            

In [ ]:
source_directory = "D:\\good\\archive(2)"
destination_directory = "C:\\Users\\aa886\\OneDrive\\Desktop\\data"
rename_pattern = "image_{count}.{extention}"
shutil.rmtree(destination_directory+"\\images")
shutil.rmtree(destination_directory+"\\labels")
export_dataset(source_directory,destination_directory,rename_pattern,number_of_samples_needed=200,validation=0.2)


[3, 9, 12, 21, 25, 26, 30, 37, 38, 40, 41, 47, 50, 54, 60, 66, 68, 75, 76, 78, 79, 84, 86, 92, 94, 114, 127, 129, 134, 136, 152, 155, 157, 161, 172, 177, 180, 184, 191, 196] 40
[200, 206, 208, 219, 222, 226, 238, 241, 253, 255, 260, 268, 273, 277, 281, 287, 288, 301, 313, 318, 323, 328, 329, 332, 334, 337, 349, 350, 353, 357, 363, 366, 367, 377, 381, 384, 390, 392, 398, 399] 40
[400, 401, 402, 405, 407, 408, 409, 421, 426, 432, 437, 443, 447, 454, 458, 462, 477, 479, 488, 491, 495, 499, 515, 519, 523, 524, 537, 546, 547, 551, 555, 558, 563, 570, 571, 577, 580, 584, 587, 594] 40
